# Customer Shopping Behaviour — Feature Engineering

**Project:** End-to-End Retail Customer Behaviour Analytics  
**Author:** Daniel Sampson  
**Tools:** Python (Pandas), SQLAlchemy, pyodbc

---

## Objective
Engineer new features from the cleaned dataset and load
the final DataFrame into Microsoft SQL Server for business
question analysis.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

# Load the cleaned dataset
df = pd.read_csv('../data/customer_data_cleaned.csv')

print(f'Cleaned dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')

Cleaned dataset loaded: 3900 rows, 17 columns


## 1. Age Group (Binning)

Bin continuous age values into four demographic segments
using quantile-based discretisation to ensure equal group sizes.

In [2]:
# Create age_group column using quartile-based bins
labels = ['Young Adult', 'Adult', 'Middle Aged', 'Senior']

df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)

# Verify the distribution
print('Age group distribution:')
print(df['age_group'].value_counts().sort_index())

Age group distribution:
age_group
Young Adult    1028
Adult           942
Middle Aged     986
Senior          944
Name: count, dtype: int64


## 2. Purchase Frequency Days

Convert the categorical `frequency_of_purchases` column
into a numeric value (days between purchases) to enable
quantitative comparisons and analysis.

In [3]:
# Map frequency categories to approximate number of days
frequency_mapping = {
    'Weekly'        : 7,
    'Fortnightly'   : 14,
    'Bi-Weekly'     : 14,
    'Monthly'       : 30,
    'Quarterly'     : 90,
    'Every 3 Months': 90,
    'Annually'      : 365
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

# Check for any unmapped values (NaN means a category was missed)
print('Unmapped frequency values:')
print(df[df['purchase_frequency_days'].isna()]['frequency_of_purchases'].unique())

# Verify the new column
print('\nSample output:')
print(df[['frequency_of_purchases', 'purchase_frequency_days']].drop_duplicates())

Unmapped frequency values:
[]

Sample output:
   frequency_of_purchases  purchase_frequency_days
0             Fortnightly                       14
2                  Weekly                        7
4                Annually                      365
6               Quarterly                       90
10              Bi-Weekly                       14
15                Monthly                       30
20         Every 3 Months                       90


## 3. Verify Final Dataset

In [4]:
# Final shape and column list
print(f'Final shape: {df.shape}')
print(f'\nColumns ({df.shape[1]}):')
print(df.columns.tolist())

Final shape: (3900, 19)

Columns (19):
['customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'shipping_type', 'discount_applied', 'previous_purchases', 'payment_method', 'frequency_of_purchases', 'age_group', 'purchase_frequency_days']


In [5]:
# Final shape and column list
print(f'Final shape: {df.shape}')
print(f'\nColumns ({df.shape[1]}):')
print(df.columns.tolist())

Final shape: (3900, 19)

Columns (19):
['customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'shipping_type', 'discount_applied', 'previous_purchases', 'payment_method', 'frequency_of_purchases', 'age_group', 'purchase_frequency_days']


In [6]:
# 1. Defining my exact connection parameters from SSMS
SERVER = r'DESKTOP-JJFIAGN\SQLEXPRESS'
DATABASE = 'Customer_Behaviour'
DRIVER = 'ODBC Driver 17 for SQL Server' 

# 2. Build the connection string using Windows Authentication
connection_string = f'DRIVER={{{DRIVER}}};SERVER={SERVER};DATABASE={DATABASE};Trusted_Connection=yes;'


# 3. URL-encode the string for SQLAlchemy
params = urllib.parse.quote_plus(connection_string)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# 4. Test the connection
try:
    with engine.connect() as connection:
        print("Successfully connected to MS SQL Server!")
except Exception as e:
    print(f"Connection failed: {e}")

Successfully connected to MS SQL Server!


In [7]:
# Load the final DataFrame into SQL Server
# if_exists='replace' drops and recreates the table each run
df.to_sql(
    name      = 'customer_data',
    con       = engine,
    if_exists = 'replace',
    index     = False
)

print("Table 'customer_data' successfully loaded into SQL Server.")
print(f'Rows loaded: {len(df)}')

Table 'customer_data' successfully loaded into SQL Server.
Rows loaded: 3900


## 5. Summary of Engineered Features

| Feature | Method | Purpose |
|---|---|---|
| `age_group` | pd.qcut — quartile binning | Demographic segmentation |
| `purchase_frequency_days` | Dictionary mapping | Quantitative frequency analysis |

## 6. Pipeline Complete

| Step | File | Status |
|---|---|---|
| EDA | 01_eda.ipynb | ✅ Done |
| Data Cleaning | 02_data_cleaning.ipynb | ✅ Done |
| Feature Engineering + SQL Load | 03_feature_engineering.ipynb | ✅ Done |

---
*Next: `sql/customer_behavior_sql_queries.sql` — Business question analysis in SQL Server*